# 第三章 迭代优化

当使用 LLM 构建应用程序时，实践层面上很难*第一次尝试*就成功获得适合最终应用的 Prompt。但这并不重要，只要您有一个好的迭代过程来不断改进您的 Prompt，那么您就能够得到一个适合任务的 Prompt。虽然相比训练机器学习模型，在  Prompt 方面一次成功的几率可能会高一些，但正如上所说， Prompt 是否一次完善并不重要。最重要的是**层层迭代**为您的应用程序找到有效  Prompt 的过程。

因此在本章中，我们将以产品说明书中生成营销文案为例，来展示一些流程框架，并提示您思考如何层层迭代地分析和完善您的 Prompt。

在吴恩达（Andrew Ng，原教程作者）的机器学习课程中展示过一张图表，说明了机器学习开发的流程。通常是先有一个想法，然后再用以下流程实现：编写代码，获取数据，训练模型，获得实验结果。然后您可以查看结果，分析误差与错误，找出适用领域，甚至可以更改您对具体问题的具体思路或解决方法。此后再次更改实现，并运行另一个实验等，反复迭代，最终获得有效的机器学习模型。在编写基于 LLM 的应用程序的 Prompt 时，流程可能非常相似。您产生了关于要完成的任务的想法后，可以尝试编写第一个 Prompt ，注意要满足上一章说过的两个原则：**清晰明确，并且给系统足够的时间思考**。然后您可以运行并查看结果。如果第一次效果不好，那么迭代的过程就是找出为什么指令不够清晰或为什么没有给算法足够的时间思考，以便改进想法、改进  Prompt 等等，循环多次，直到找到适合您的应用程序的 Prompt。

很难有适用于世间万物的所谓“最佳  Prompt ”，更好的方法是找到有效的迭代过程，以便您可以快速地找到一个适合您的应用程序的  Prompt 。


<div class="toc">
    <ul class="toc-item">
        <li><span><a href="#一环境配置" data-toc-modified-id="一、环境配置">一、环境配置</a></span></li>
        <li>
            <span><a href="#二任务从产品说明书生成一份营销产品描述" data-toc-modified-id="二、任务——从产品说明书生成一份营销产品描述">二、任务——从产品说明书生成一份营销产品描述</a></span>
            <ul class="toc-item">
                <li><span><a href="#21-问题一生成文本太长" data-toc-modified-id="2.1 问题一：生成文本太长">2.1 问题一：生成文本太长</a></span></li>
                <li><span><a href="#22-问题二抓错文本细节" data-toc-modified-id="2.2 问题二：抓错文本细节">2.2 问题二：抓错文本细节</a></span></li>
                <li><span><a href="#23-问题三添加表格描述" data-toc-modified-id="2.3 问题三：添加表格描述">2.3 问题三：添加表格描述</a></span></li>
            </ul>
        </li>
    </ul>
</div>

## 一、环境配置

同上一章，我们首先需要配置使用 OpenAI API 的环境

In [32]:
import os
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv
from IPython.display import Markdown
# 导入第三方库

loaded = load_dotenv(find_dotenv(), override=True)
API_KEY = os.getenv("API_KEY")

# 如果您使用的是官方 API，就直接用 https://api.siliconflow.cn/v1 就行。
BASE_URL = "https://api.siliconflow.cn/v1"

In [33]:
# 实例化 OpenAI 对象
# 传入参数：OpenAI API Key（必需）、Base URL 和最大重试次数
client = OpenAI(api_key=API_KEY, base_url=BASE_URL, max_retries=3)

In [34]:
# 参数 n，整数或 Null，可选项，默认为 1。为每条输入信息生成多少个聊天完成选项。
# 参数 temperature，实数值或 Null，可选项，默认为 1。使用的采样温度，介于 0 和 2 之间。0.8 等较高值会使输出更加随机，而 0.2 等较低值会使输出更加集中和确定。

def get_completions(llm_prompt, model_endpoint):
    extra_body = {}
    if "Qwen3" in model_endpoint:
        extra_body={
            "enable_thinking": False
        }
        
    response = client.chat.completions.create(model=model_endpoint,
                                              messages=[
                                                        {"role": "user",
                                                         "content": llm_prompt
                                                        }
                                                       ],
                                              n=1, temperature=0, seed=42,
                                              presence_penalty=0, frequency_penalty=0,
                                              max_tokens=512, extra_body = extra_body
                                             )

    return response.choices[0].message.content.strip()

## 二、任务——从产品说明书生成一份营销产品描述

给定一份椅子的资料页。描述说它属于*中世纪灵感*系列，产自意大利，并介绍了材料、构造、尺寸、可选配件等参数。假设您想要使用这份说明书帮助营销团队为电商平台撰写营销描述稿：

In [56]:
llm = "Qwen/Qwen3-8B"

# 示例：产品说明书
fact_sheet_chair_en = """
OVERVIEW
- Part of a beautiful family of mid-century inspired office furniture, 
including filing cabinets, desks, bookcases, meeting tables, and more.
- Several options of shell color and base finishes.
- Available with plastic back and front upholstery (SWC-100) 
or full upholstery (SWC-110) in 10 fabric and 6 leather options.
- Base finish options are: stainless steel, matte black, 
gloss white, or chrome.
- Chair is available with or without armrests.
- Suitable for home or business settings.
- Qualified for contract use.

CONSTRUCTION
- 5-wheel plastic coated aluminum base.
- Pneumatic chair adjust for easy raise/lower action.

DIMENSIONS
- WIDTH 53 CM | 20.87”
- DEPTH 51 CM | 20.08”
- HEIGHT 80 CM | 31.50”
- SEAT HEIGHT 44 CM | 17.32”
- SEAT DEPTH 41 CM | 16.14”

OPTIONS
- Soft or hard-floor caster options.
- Two choices of seat foam densities: 
medium (1.8 lb/ft3) or high (2.8 lb/ft3)
- Armless or 8 position PU armrests 

MATERIALS
SHELL BASE GLIDER
- Cast Aluminum with modified nylon PA6/PA66 coating.
- Shell thickness: 10 mm.
SEAT
- HD36 foam

COUNTRY OF ORIGIN
- Italy
"""

In [36]:
llm = "Qwen/Qwen3-8B"

#   Prompt ：基于说明书生成营销描述
prompt = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

Technical specifications: ```{fact_sheet_chair_en}```
"""
response = get_completions(prompt, llm)
print(response)


Introducing the **SWC-100/110 Mid-Century Inspired Office Chair** — a timeless piece designed to blend seamlessly into both modern and traditional workspaces. Inspired by the elegant aesthetics of the mid-century era, this chair is part of a beautiful family of office furniture that includes filing cabinets, desks, and more, making it a perfect match for any office or home décor.

Choose from a variety of shell color and base finish options to customize your chair to your personal style or office environment. The base is available in **stainless steel, matte black, gloss white, or chrome**, each offering a unique look and feel. For added comfort, the chair comes with **plastic back and front upholstery (SWC-100)** or **full upholstery (SWC-110)** in **10 fabric and 6 leather options**, ensuring both style and durability.

Featuring a **5-wheel plastic coated aluminum base**, this chair is built for smooth movement and stability across different floor types. The **pneumatic adjust syste

In [37]:
# 示例：产品说明书
fact_sheet_chair_zh = """
概述

    美丽的中世纪风格办公家具系列的一部分，包括文件柜、办公桌、书柜、会议桌等。
    多种外壳颜色和底座涂层可选。
    可选塑料前后靠背装饰（SWC-100）或10种面料和6种皮革的全面装饰（SWC-110）。
    底座涂层选项为：不锈钢、哑光黑色、光泽白色或铬。
    椅子可带或不带扶手。
    适用于家庭或商业场所。
    符合合同使用资格。

结构

    五个轮子的塑料涂层铝底座。
    气动椅子调节，方便升降。

尺寸

    宽度53厘米|20.87英寸
    深度51厘米|20.08英寸
    高度80厘米|31.50英寸
    座椅高度44厘米|17.32英寸
    座椅深度41厘米|16.14英寸

选项

    软地板或硬地板滚轮选项。
    两种座椅泡沫密度可选：中等（1.8磅/立方英尺）或高（2.8磅/立方英尺）。
    无扶手或8个位置PU扶手。

材料
外壳底座滑动件

    改性尼龙PA6/PA66涂层的铸铝。
    外壳厚度：10毫米。
    座椅
    HD36泡沫

原产国

    意大利
"""

In [38]:
llm = "Qwen/Qwen3-8B"

#   Prompt ：基于说明书创建营销描述
prompt = f"""
您的任务是帮助营销团队基于技术说明书创建一个产品的营销描述。

根据```标记的技术说明书中提供的信息，编写一个产品描述。

技术说明: ```{fact_sheet_chair_zh}```
"""
response = get_completions(prompt, llm)
print(response)


当然！以下是基于您提供的技术说明书内容，为该产品撰写的营销描述：

---

**优雅与实用的完美结合 —— 中世纪风格办公家具系列**

融入中世纪风格的灵感，这款办公家具系列为现代办公空间带来独特的艺术气息与历史韵味。无论是用于家庭书房还是商务办公室，它都能完美融入各种环境，提升整体格调与舒适度。

**经典设计，现代功能**

- **坚固耐用的铸铝底座**：采用改性尼龙PA6/PA66涂层的铸铝材质，不仅增强了结构的稳定性，还赋予其出色的抗腐蚀性能。底座厚度达10毫米，确保长期使用不变形。
- **五轮塑料涂层底座**：配备五轮滚轮系统，提供灵活移动能力，同时可选择软地板或硬地板滚轮，适应不同地面材质，提升使用体验。
- **气动椅调节系统**：轻松升降，满足不同身高需求，让您的办公体验更加人性化。

**舒适座椅，多种选择**

- **HD36泡沫座椅**：提供卓越的支撑性和舒适度，适合长时间办公。
- **座椅高度与深度**：分别为44厘米（17.32英寸）和41厘米（16.14英寸），确保人体工学设计，贴合您的身体曲线。
- **扶手选项**：可选择无扶手或8个位置PU扶手，满足不同使用习惯与空间需求。

**个性化定制，彰显品味**

- **外壳颜色与底座涂层**：提供多种外壳颜色选择，底座可选不锈钢、哑光黑色、光泽白色或铬，满足您对风格与质感的不同追求。
- **装饰选项**：可选配塑料前后靠背装饰（SWC-100）或10种面料与6种皮革的全面装饰（SWC-110），让您的办公家具更具个性与奢华感。

**品质源自意大利**

这款产品源自意大利，以其精湛的工艺和对细节的执着追求，成为高品质办公家具的代表。符合合同使用资格，是您打造专业办公环境的理想之选。

---

如需进一步定制或了解产品详情，欢迎随时联系我们的销售团队。


## 2.1 问题一：生成文本太长

它似乎很好地完成了要求，即从技术说明书开始编写产品描述，介绍了一个精致的中世纪风格办公椅。但是当我看到这个时，我会觉得这个太长了。

所以在上述过程中，我产生想法后写了一个  Prompt ，并得到了结果，但是我对它不是很满意，因为它太长了。所以我澄清我的  Prompt ，要求它限制生成文本长度，要求最多使用50个字。


In [51]:
llm = "Qwen/Qwen3-8B"

# 优化后的 Prompt，要求生成描述不多于 50 词
prompt = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

Use at most 50 words.

Technical specifications: ```{fact_sheet_chair_en}```
"""
response = get_completions(prompt, llm)
print(response)


Elegant mid-century inspired office chair with a 53 cm width and 80 cm height. Available in 10 fabric or 6 leather options, with or without PU armrests. Durable cast aluminum base in stainless steel, matte black, gloss white, or chrome. Perfect for home or business use.


提取回答并根据空格拆分，答案为46个字，较好地完成了设计要求。

In [52]:
lst = response.split()
print(len(lst))

46


In [41]:
llm = "Qwen/Qwen3-8B"

# 优化后的 Prompt，要求生成描述不多于 50 词
prompt = f"""
您的任务是帮助营销团队基于技术说明书创建一个产品的零售网站描述。

根据```标记的技术说明书中提供的信息，编写一个产品描述。

使用最多50个词。

技术规格：```{fact_sheet_chair_zh}```
"""
response = get_completions(prompt, llm)
print(response)


这款中世纪风格办公椅，采用铸铝底座与改性尼龙涂层，配气动调节，可选PU扶手与多种面料。优雅设计，适合家庭或商业使用，意大利制造，品质卓越。


In [42]:
# 由于中文需要分词，此处直接计算整体长度
len(response)

69

LLM在能堪堪胜任严格的字数限制，但实现得并不精确。此例中，英文输出要求控制在50个词，但有时会输出60或65个单词的内容，但这也还算合理。原因是 LLM 使用分词器（tokenizer）解释文本，但它们往往在计算字符方面表现一般般。有很多不同的方法来尝试控制您得到的输出的长度（如若干句话/词/个汉字/个字母 (characters) 等）。

## 2.2 问题二：抓错文本细节

我们继续完善这段推广词，会发现的第二个问题是，这个网站并不是直接向消费者销售，它实际上面向的是家具零售商，他们会更关心椅子的技术细节和材料。在这种情况下，您可以继续修改这个  Prompt ，让它更精确地描述椅子的技术细节。

解决方法：要求它专注于与目标受众相关的方面。

In [43]:
llm = "Qwen/Qwen3-8B"

# 优化后的 Prompt，说明面向对象，应具有什么性质且侧重于什么方面
prompt = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

The description is intended for furniture retailers, 
so should be technical in nature and focus on the 
materials the product is constructed from.

Use at most 50 words.

Technical specifications: ```{fact_sheet_chair_en}```
"""
response = get_completions(prompt, llm)
print(response)

Crafted from cast aluminum with modified nylon coating, this mid-century inspired chair features a 10 mm shell base glider for durability. The HD36 foam seat offers superior comfort, available in medium or high density. Suitable for home or contract use, it comes in multiple color and finish options.


In [44]:
llm = "Qwen/Qwen3-8B"

# 优化后的 Prompt，说明面向对象，应具有什么性质且侧重于什么方面
prompt = f"""
您的任务是帮助营销团队基于技术说明书创建一个产品的零售网站描述。

根据```标记的技术说明书中提供的信息，编写一个产品描述。

该描述面向家具零售商，因此应具有技术性质，并侧重于产品的材料构造。

使用最多50个单词。

技术规格： ```{fact_sheet_chair_zh}```
"""
response = get_completions(prompt, llm)
print(response)

这款中世纪风格办公椅采用铸铝底座，覆有改性尼龙PA6/PA66涂层，确保耐用与美观。座椅填充HD36泡沫，提供舒适支撑。可选无扶手或8位置PU扶手，适合家庭及商业环境使用。


可见，通过修改  Prompt ，模型的关注点倾向了具体特征与技术细节。

我可能进一步想要在描述的结尾展示出产品ID。因此，我可以进一步改进这个  Prompt ，要求在描述的结尾，展示出说明书中的7位产品ID。

In [58]:
llm = "Qwen/Qwen3-8B"

# 更进一步，要求在描述末尾包含 7个字符的产品ID
prompt = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

The description is intended for furniture retailers, 
so should be technical in nature and focus on the 
materials the product is constructed from.

At the end of the description, include every 7-character 
Product ID in the technical specification.

Use at most 50 words.

Technical specifications: ```{fact_sheet_chair_en}```
"""
response = get_completions(prompt, llm)
print(response)

Crafted from cast aluminum with modified nylon PA6/PA66 coating, this chair features a 10 mm shell thickness for durability. The HD36 foam seat offers superior comfort, available in medium or high density. Designed for both home and commercial use, it includes 5-wheel plastic coated base and options for stainless steel, matte black, gloss white, or chrome finishes. Product IDs: SWC-100, SWC-110.


In [62]:
llm = "Qwen/Qwen3-8B"

# 更进一步
prompt = f"""
您的任务是帮助营销团队基于技术说明书创建一个产品的零售网站描述。

根据```标记的技术说明书中提供的信息，编写一个产品描述。

该描述面向家具零售商，因此应具有技术性质，并侧重于产品的材料构造。

在描述末尾，包括技术规格中两个个7个字符的产品ID。

使用最多50个单词。

技术规格： ```{fact_sheet_chair_zh}```
"""
response = get_completions(prompt, llm)
print(response)

这款中世纪风格办公椅采用铸铝底座，表面覆有改性尼龙PA6/PA66涂层，增强耐用性与美观度。座椅填充HD36泡沫，提供舒适支撑。可选配软或硬地板滚轮，以及中等或高密度座椅泡沫。扶手可选无或8个位置PU扶手。产品ID：SWC-100、SWC-110。


以上是许多开发人员通常会经历的  Prompt 开发的迭代过程简短示例。我的建议是，像上一章中所演示的那样，Prompt 应该保持清晰和明确，并在必要时给模型一些思考时间。在这些要求的基础上，常见流程是首先尝试编写一版 Prompt ，看看会发生什么，然后继续迭代完善 Prompt，以逐渐接近所需的结果。许多成功的 Prompt 都是通过这种迭代过程得出的。我将向您展示一个更复杂的 Prompt 示例，可能会让您对 ChatGPT 的能力有更深入的了解。

## 2.3 问题三：添加表格描述
继续添加指引，要求提取产品尺寸信息并组织成表格，并指定表格的列、表名和格式；再将所有内容格式化为可以在网页使用的 HTML。

In [47]:
llm = "Qwen/Qwen3-8B"

# 要求它抽取信息并组织成表格，并指定表格的列、表名和格式
prompt = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

The description is intended for furniture retailers, 
so should be technical in nature and focus on the 
materials the product is constructed from.

At the end of the description, include every 7-character 
Product ID in the technical specification.

After the description, include a table that gives the 
product's dimensions. The table should have two columns.
In the first column include the name of the dimension. 
In the second column include the measurements in inches only.

Give the table the title 'Product Dimensions'.

Format everything as HTML that can be used in a website. 
Place the description in a <div> element.

Technical specifications: ```{fact_sheet_chair_en}```
"""

response = get_completions(prompt, llm)
print(response)

```html
<div>
  <h2>Product Description</h2>
  <p>This chair is part of a beautiful family of mid-century inspired office furniture, designed to complement a variety of workspaces including filing cabinets, desks, bookcases, meeting tables, and more. Constructed with a durable 5-wheel plastic coated aluminum base, it offers smooth and quiet movement across different floor types. The chair features a pneumatic adjust mechanism, allowing for effortless height adjustment to suit individual preferences and ergonomic needs.</p>
  
  <p>The chair's frame is made from cast aluminum with a modified nylon PA6/PA66 coating, providing enhanced corrosion resistance and a sleek, modern finish. The shell base glider is engineered with a shell thickness of 10 mm, ensuring stability and long-lasting performance. The seat is upholstered with HD36 foam, offering superior comfort and support. Available in several options of shell color and base finishes, including stainless steel, matte black, gloss whit

In [48]:
# 表格是以 HTML 格式呈现的，加载出来
from IPython.display import display, HTML

display(HTML(response))

Dimension,Measurements (inches)
WIDTH,20.87
DEPTH,20.08
HEIGHT,31.50


In [49]:
llm = "Qwen/Qwen3-8B"

# 要求它抽取信息并组织成表格，并指定表格的列、表名和格式
prompt = f"""
您的任务是帮助营销团队基于技术说明书创建一个产品的零售网站描述。

根据```标记的技术说明书中提供的信息，编写一个产品描述。

该描述面向家具零售商，因此应具有技术性质，并侧重于产品的材料构造。

在描述末尾，包括技术规格中每个7个字符的产品ID。

在描述之后，包括一个表格，提供产品的尺寸。表格应该有两列。第一列包括尺寸的名称。第二列只包括英寸的测量值。

给表格命名为“产品尺寸”。

将所有内容格式化为可用于网站的HTML格式。将描述放在<div>元素中。

技术规格：```{fact_sheet_chair_zh}```
"""

response = get_completions(prompt, llm)
print(response)

```html
<div>
  <h2>中世纪风格办公椅 - 现代设计与经典工艺的完美结合</h2>
  <p>这款办公椅是中世纪风格办公家具系列中的经典之作，专为追求品质与风格的用户打造。其设计融合了传统工艺与现代功能性，适用于家庭办公或商业环境。</p>
  <p>椅子采用五轮塑料涂层铝底座，确保稳定性和耐用性。底座表面经过改性尼龙PA6/PA66涂层处理，具有出色的抗磨损和抗腐蚀性能。外壳厚度为10毫米，增强了整体结构的坚固性。</p>
  <p>座椅部分采用HD36高密度泡沫，提供卓越的舒适度与支撑性，长时间使用也不会塌陷。气动调节系统允许用户轻松调整座椅高度，以适应不同的身高需求。</p>
  <p>该产品提供多种定制选项，包括软地板或硬地板滚轮，以及两种座椅泡沫密度：中等（1.8磅/立方英尺）或高（2.8磅/立方英尺）。此外，扶手可选无扶手或8个位置PU扶手，满足不同用户的使用偏好。</p>
  <p>外壳颜色和底座涂层可选，包括不锈钢、哑光黑色、光泽白色或铬，为用户提供了丰富的个性化选择。</p>
  <p>本产品符合合同使用资格，是办公空间中兼具美观与实用性的理想选择。</p>
  <p>产品ID: SWC-100</p>
  <p>产品ID: SWC-110</p>
</div>

<table>
  <caption>产品尺寸</caption>
  <tr>
    <th>尺寸名称</th>
    <th>英寸测量值</th>
  </tr>
  <tr>
    <td>宽度</td>
    <td>20.87</td>
  </tr>
  <tr>
    <td>深度</td>
    <td>20.08</td>
  </tr>
  <tr>
    <td>高度</td>
    <td>31.50</td>
  </tr>
  <tr>
    <td>座椅高度</td>
    <td>17.32</td>
  </tr>
  <tr>


In [50]:
# 表格是以 HTML 格式呈现的，加载出来
from IPython.display import display, HTML

display(HTML(response))

尺寸名称,英寸测量值
宽度,20.87
深度,20.08
高度,31.50
座椅高度,17.32


本章的主要内容是 LLM 在开发应用程序中的迭代式  Prompt 开发过程。开发者需要先尝试编写  Prompt ，然后通过迭代逐步完善它，直至得到需要的结果。作为一名高效的提示词工程师（Prompt Engineer），关键在于掌握有效的开发Prompt的过程，而不是去寻求得到“完美的”Prompt。对于一些更复杂的应用程序，可以对多个样本（如数百张说明书）进行  Prompt 的迭代开发，并在样本集上进行评估。

最后，在更成熟的应用程序中，可以观察多个Prompt在多个样本集上的表现，测试平均或最差性能。但通常，**仅当**应用较成型之后，才推荐您通过这种评估方式，来精益求精。

请使用 Jupyter Notebook，动手实践本节给出的示例，并尝试不同的变化，查看结果。